# PatchScout on Google Colab

Run the full PatchScout pipeline on Colab — including the local DeepSeek Coder 6.7B Instruct model — without a local GPU.

**What this notebook does**
1. Installs Ollama and pulls `deepseek-coder:6.7b-instruct` onto the Colab GPU.
2. Clones (or unzips) the PatchScout repo and installs the runtime dependencies.
3. Runs the CLI against the bundled `test_samples/` in three detection modes (`pattern`, `deepseek`, `hybrid`).
4. Produces a benchmark CSV + plot you can paste straight into your research paper.
5. (Optional) Exposes the Flask web UI through a Cloudflare quick tunnel.

**Before you start**  
Runtime → Change runtime type → **GPU (T4)**. The 6.7B model needs ~4 GB of VRAM (Q4-quantized) so the free tier T4 (15 GB) is enough.

**Approximate setup time:** 5–8 minutes (Ollama install + model pull).

## 0. Sanity check — GPU is attached

In [ ]:
!nvidia-smi

## 1. Install Ollama and start the server

Ollama is the local inference server PatchScout's `DeepSeekRunner` talks to. The install script drops a Linux binary into `/usr/local/bin`.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests

# Start Ollama in the background and wait for its HTTP server
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT,
    env={**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434'},
)

for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            print('Ollama is up.')
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama failed to start — check /tmp/ollama.log')

### Pull the model (one-time, ~3.8 GB)

This is the slow cell — typically 2–4 minutes on Colab. The model is cached in `~/.ollama` for the lifetime of the runtime.

In [ ]:
!ollama pull deepseek-coder:6.7b-instruct

In [ ]:
# Smoke test the model before we wire it into PatchScout
import requests
r = requests.post(
    'http://127.0.0.1:11434/api/chat',
    json={
        'model': 'deepseek-coder:6.7b-instruct',
        'messages': [{'role': 'user', 'content': 'Reply with the single word: READY'}],
        'stream': False,
        'options': {'num_predict': 10, 'temperature': 0.0},
    },
    timeout=120,
)
print(r.json()['message']['content'])

## 2. Get PatchScout

Pick **one** of the two options below.

### Option A — clone from GitHub (recommended if your repo is public)

In [ ]:
!git clone https://github.com/parthpatil7/patchscout.git PatchScout
%cd PatchScout

### Option B — upload a ZIP from your laptop
Run this *instead* of Option A if your repo is private. Comment-uncomment as needed.

In [ ]:
# from google.colab import files
# import zipfile, os
# uploaded = files.upload()                  # pick PatchScout.zip
# zip_name = next(iter(uploaded))
# zipfile.ZipFile(zip_name).extractall('.')
# # If the zip extracts to a folder, cd into it:
# # %cd PatchScout

### Install Python dependencies

PatchScout's `requirements.txt` pins heavyweight packages (torch, transformers, clang, semgrep) that aren't needed on Colab — DeepSeek runs through Ollama over HTTP, and the static detector uses regex + tree-sitter. We install only the runtime subset:

In [ ]:
!pip install -q rich pyyaml pandas openpyxl requests flask pyngrok matplotlib tree-sitter-languages

## 3. Sanity run — CLI on the bundled vulnerable samples

`test_samples/` ships with intentionally vulnerable C, Java, Python and PHP files. We'll scan them three times — once per detection mode — to baseline the tool.

In [ ]:
!ls -la test_samples/

### 3a. Pattern-only (no LLM — fastest baseline)

In [ ]:
import yaml, pathlib
cfg_path = pathlib.Path('config/config.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['detection']['engine'] = 'pattern'
cfg_path.write_text(yaml.dump(cfg))

!python -m src.main -d test_samples -o output/pattern_only.xlsx -v

### 3b. DeepSeek-only (LLM verdict + fix)

First file takes ~30–60 s while the model loads into VRAM; subsequent files are faster.

In [ ]:
cfg['detection']['engine'] = 'deepseek'
cfg_path.write_text(yaml.dump(cfg))

!python -m src.main -d test_samples -o output/deepseek.xlsx -v --remediation

### 3c. Hybrid (weighted fusion — what the paper should evaluate)

In [ ]:
cfg['detection']['engine'] = 'hybrid'
cfg_path.write_text(yaml.dump(cfg))

!python -m src.main -d test_samples -o output/hybrid.xlsx -v --remediation

## 4. Benchmark harness for the paper

Re-runs every test file in all three modes inside a Python loop so we can capture per-file timings, severity distributions, and detection-method tags (`both` / `static_only` / `llm_only` / `anomaly`).  Outputs a CSV + PNG you can drop into the paper.

In [ ]:
import json, time, sys
from copy import deepcopy
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.analyzers import CodeAnalyzer
from src.utils import ConfigLoader

# Reset the loader so the engine override is read fresh each iteration
base_cfg = ConfigLoader().load_config()
samples = sorted(p for p in Path('test_samples').iterdir() if p.is_file())
modes = ['pattern', 'deepseek', 'hybrid']

rows = []
for mode in modes:
    cfg = deepcopy(base_cfg)
    cfg['detection']['engine'] = mode
    analyzer = CodeAnalyzer(cfg)
    for f in samples:
        t0 = time.time()
        r = analyzer.analyze_file(str(f))
        dt = time.time() - t0
        vulns = r.get('vulnerabilities', [])
        sev = {'Critical': 0, 'High': 0, 'Medium': 0, 'Low': 0}
        tags = {'static_only': 0, 'both': 0, 'llm_only': 0, 'anomaly': 0}
        for v in vulns:
            s = v.get('severity', 'Medium')
            sev[s] = sev.get(s, 0) + 1
            t = v.get('detection_method', 'static_only')
            tags[t] = tags.get(t, 0) + 1
        rows.append({
            'mode': mode,
            'file': f.name,
            'language': r.get('language'),
            'lines_of_code': r.get('lines_of_code'),
            'time_s': round(dt, 2),
            'total_vulns': len(vulns),
            **sev,
            **tags,
        })
        print(f'{mode:8s}  {f.name:25s}  vulns={len(vulns):3d}  time={dt:5.2f}s')

df = pd.DataFrame(rows)
Path('output').mkdir(exist_ok=True)
df.to_csv('output/benchmark.csv', index=False)
df

### Plot — vulnerabilities found vs runtime, per mode

In [ ]:
import matplotlib.pyplot as plt

agg = df.groupby('mode')[['total_vulns', 'time_s']].sum().reindex(modes)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
agg['total_vulns'].plot.bar(ax=ax[0], color='steelblue')
ax[0].set_title('Total vulnerabilities flagged')
ax[0].set_ylabel('count')
agg['time_s'].plot.bar(ax=ax[1], color='darkorange')
ax[1].set_title('Total runtime')
ax[1].set_ylabel('seconds')
plt.tight_layout()
plt.savefig('output/benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

### Detection-method breakdown (hybrid mode only)

This is the headline chart for the fusion architecture — how many findings came from static-only, LLM-only, the overlap, or anomalies.

In [ ]:
hybrid_df = df[df['mode'] == 'hybrid'][['both', 'static_only', 'llm_only', 'anomaly']].sum()
ax = hybrid_df.plot.pie(autopct='%1.0f%%', figsize=(5, 5), ylabel='')
ax.set_title('Hybrid mode — detection method distribution')
plt.savefig('output/hybrid_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Download the reports to your laptop

Pulls every Excel report and the benchmark artefacts off Colab in one go.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('patchscout_results', 'zip', 'output')
files.download('patchscout_results.zip')

## 6. (Optional) Run the Flask web UI behind a Cloudflare tunnel

Lets you drive the upload page from your laptop browser while inference runs on the Colab GPU. The tunnel URL ends in `.trycloudflare.com`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, re, pathlib

flask_proc = subprocess.Popen(
    ['python', 'web_app.py'],
    stdout=open('/tmp/flask.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=open('/tmp/tunnel.log', 'w'),
    stderr=subprocess.STDOUT,
)

url = None
for _ in range(30):
    time.sleep(1)
    log = pathlib.Path('/tmp/tunnel.log').read_text()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m:
        url = m.group(0)
        break
print('Web UI:', url or 'not ready yet — re-run this cell')

## 7. Shut down

Colab will reclaim the VM after idle timeout, but you can free RAM/VRAM explicitly:

In [ ]:
# Stop Flask + tunnel + Ollama
!pkill -f 'python web_app.py' || true
!pkill -f cloudflared || true
!pkill -f 'ollama serve' || true

---

## Notes for the research paper

- **Reproducibility.** The benchmark cell pins the engine via `config.yaml` and uses `deepseek-coder:6.7b-instruct` with `temperature=0.1`, `num_ctx=4096`, `num_predict=1024` (set in `src/ml/deepseek_runner.py`). Record the Colab GPU model from cell 0 — T4 vs A100 changes runtime numbers materially.
- **Ground truth.** `test_samples/` are intentionally vulnerable, so the **recall** side is what's measurable here. For **precision** you'll want to also run the tool against a known-clean corpus (e.g. small, well-reviewed projects) and treat every finding there as a false positive.
- **Fusion metric.** The `both` count is the strongest signal — it's where static and LLM agree on the same CWE. `llm_only` shows where DeepSeek adds value beyond regex; `static_only` shows where the LLM missed a known pattern.
- **Suggested experiments.** (a) Sweep `ml_threshold` (0.2 → 0.6) and plot precision/recall vs threshold. (b) Compare hybrid against pattern-only on a bigger benchmark like Juliet or SARD. (c) Measure latency per LOC.